In [1]:
from mydatasets.BaseDatabase import BaseDataset
from agents.doc_quest import DocQuestAgents
import toml  # type: ignore[import-untyped]

toml_cfg_path="/home/cjy/DocQuest/DocQuest/config/doc_quest_config.toml"
with open(toml_cfg_path, "r") as f:
        dq_cfg = toml.load(f)
run_name="ipynb"
dataset_name="fsl"
dq_cfg["run-name"] = run_name
dataset = BaseDataset(dq_cfg, dataset_name)
docQuestAgents = DocQuestAgents(dq_cfg)
sample={
        "doc_id": "0e94b4197b10096b1f4c699701570fbf.pdf",
        "doc_type": "Tutorial/Workshop",
        "question": "According to the chart on page 14 how much time was spent with family and friends in 2010?",
        "answer": "21%",
        "evidence_pages": "[14]",
        "evidence_sources": "['Chart']",
        "answer_format": "Float",
        "retrieval-query": "According to the chart on page 14, some amount of time was spent with family and friends in 2010 for unspecified reasons.",
        "retrieval-key": "chart, page 14, time spent, family and friends, 2010",
        "qwen_retrieval-key": "2010, time spent, family and friends",
        "qwen_retrieval-query": "According to the chart on page 14, some amount of time was spent with family and friends in 2010 for some reason.",
        "text-index-path-question": ".ragatouille/colbert/indexes/fsl-question-0e94b4197b10096b1f4c699701570fbf.pdf",
        "text-top-10-question": [
            9,
            3,
            8,
            2,
            10,
            11,
            0,
            7,
            6,
            13
        ],
        "text-top-10-question_score": [
            7.75390625,
            6.7890625,
            6.6875,
            6.4765625,
            6.37890625,
            6.35546875,
            6.21484375,
            6.01953125,
            5.89453125,
            5.8515625
        ],
        "image-top-10-question": [
            3,
            12,
            5,
            14,
            13,
            0,
            10,
            11,
            8,
            6
        ],
        "image-top-10-question_score": [
            7.047145843505859,
            5.965827465057373,
            5.930147171020508,
            5.903757572174072,
            5.773287773132324,
            5.701374053955078,
            5.699911594390869,
            5.638657569885254,
            5.435965538024902,
            5.39284086227417
        ]
    }

docQuestAgents.predict(sample,dataset)

ModuleNotFoundError: No module named 'mydatasets.BaseDatabase'

In [ ]:
from models.remote_llm import Qwen3VL
import base64
import json
vl=Qwen3VL(config=None)
image_path1="/home/cjy/DocQuest/DocQuest/tmp/pdffigure2/fsl/image/2401.18059v1-Figure7-1.png"
caption1="Figure 7: Histogram showing the percentage of nodes retrieved from different layers of the RAPTOR tree across three datasets (NarrativeQA, Quality, and Qasper) using three retrievers (SBERT, BM25, and DPR). The data indicate that a substantial portion of the nodes contributing to the final retrieval comes from non-leaf layers, with a notable percentage from the first and second layers, highlighting the importance of RAPTOR’s hierarchical summarization in the retrieval process."
with open(image_path1, "rb") as image_file:
    base64_image = base64.b64encode(image_file.read()).decode("utf-8")
image_path2="/home/cjy/DocQuest/DocQuest/tmp/pdffigure2/fsl/image/2401.18059v1-Table17-1.png"
caption2="Table 17: Performance of RAPTOR when querying different layers of the tree for Story 5."
with open(image_path2, "rb") as image_file:
    base64_image2 = base64.b64encode(image_file.read()).decode("utf-8")
message_contents=[
        {"type": "text", "text": f"here is the cation of this figure:{caption1}\n the next is the image of figure:"},
        {
            "type": "image_url",
            "image_url": {
                "url": f"data:image/png;base64,{base64_image}"
            },
        },
        {"type": "text", "text": f"here is the cation of this figure:{caption2}\n the next is the image of figure::"},
        {
            "type": "image_url",
            "image_url": {
                "url": f"data:image/png;base64,{base64_image2}"
            },
        },
        {"type": "text", "text":"describe figure 7"}
    ]
with open("./testdata.json",'r')as f:
    message=json.load(f)
output,_=vl.query(messages=message)
print(output)

According to the chart on page 14, the time spent with family and friends in 2010 was 21%.


In [ ]:
from tqdm import tqdm
import json
from models.remote_llm import Qwen3
model=Qwen3(None)

def eval(question: str, answer: str, ground_truth: str):
    r_prompt="""
Question: {question}
Predicted Answer: {answer}
Ground Truth Answer: {gt}

Please evaluate if the predicted answer is correct compared to the ground truth, considering the following criteria:

Score the answer on:
Binary correctness (0-1): 1 if the answer is correct, 0 if it is incorrect

Return only a JSON-parsable string in the format: {{"binary_correctness": <score>}}
Output:
"""
    prompt = r_prompt.format(
        question=question, answer=answer, gt=ground_truth
    )
    try:
        messages = [{"role": "user", "content": f"{prompt}"}]
        generated_ans, _ = model.query(messages=messages)
        result = extract_evaluation_metrics(generated_ans)
        return result
    except Exception as e:
        print(f"Error evaluating answer: {str(e)}")
        return {"binary_correctness": 0}


def extract_evaluation_metrics(eval_str: str) -> dict[str, float | int]:
    try:
        start_index = eval_str.find("{")
        end_index = eval_str.rfind("}") + 1
        eval_str = eval_str[start_index:end_index]
        metrics = json.loads(eval_str)
        return {"binary_correctness": int(metrics.get("binary_correctness", 0))}
    except json.JSONDecodeError as e:
        return {"binary_correctness": 0}
    except Exception as e:
        return {"binary_correctness": 0}


result="/home/cjy/DocQuest/DocQuest/results/MMLongBench/test1/2025-06-17-14-39.json"
with open(result,'r') as f:
    samples=json.load(f)
assert isinstance(samples,list)
total_score = 0.0
count = 0
max_retries = 1
for sample in tqdm(samples):
    retry_count = 0
    question = sample["question"]
    answer = sample["ans_test1"]
    gt = sample["answer"]
    while retry_count <= max_retries:
        try:
            result = eval(question, answer, gt)
            sample["binary_correctness"] = result.get("binary_correctness", 0)
            count += 1
            total_score += sample["binary_correctness"]
            break
        except Exception as e:
            retry_count += 1
            if retry_count >= max_retries:
                break
            print(f"Error evaluating sample: {str(e)}")
avg_binary_correctness = total_score / count if count > 0 else 0.0
print(avg_binary_correctness)

100%|██████████| 1073/1073 [13:36<00:00,  1.31it/s]

0.3299161230195713


# 测试文档索引

In [ ]:
from mydatasets.BaseDatabase import BaseDataset
from agents.doc_quest import IndexAgent
import toml  # type: ignore[import-untyped]

toml_cfg_path="/home/cjy/DocQuest/DocQuest/config/doc_quest_config.toml"
with open(toml_cfg_path, "r") as f:
        dq_cfg = toml.load(f)
run_name="ipynb"
dataset_name="fsl"
dq_cfg["run-name"] = run_name
summarize_prompt="""
You are tasked with performing a page-by-page summary of a document. Follow these steps:

count the pages of the document and summarize each page.if a page do not have any content to summarize, use 'have no content' to indicate.
If a page contains figures or tables, include a brief summary of them within the page's summary.
Each summary should be concise—limited to one sentence.
The final output should follow this format:
Page numbers:{number}
- Page {number}: [One-sentence summary of the page content.]
  - Figure {name}: [One-sentence summary of the figure.]

- Page {number}: [One-sentence summary of the page content.]
  - Table {name}: [One-sentence summary of the table.]

- Page {number}: have no content
...
Ensure clarity and consistency in your summaries, and maintain the structure above.

Do not drop any pages
Here is the document:
"""
page_prompt="""
"""

dataset = BaseDataset(dq_cfg, dataset_name)
ia=IndexAgent(summarize_prompt)
print(ia.index_document(dataset,"0e94b4197b10096b1f4c699701570fbf.pdf"))


In [ ]:
from mydatasets.DocumentSummarizer import DocumentSummarizer
import toml  # type: ignore[import-untyped]
from mydatasets.BaseDataset import BaseDataset

toml_cfg_path="/home/forlorin/project/DocQuest/config/doc_quest_config.toml"
with open(toml_cfg_path, "r") as f:
        dq_cfg = toml.load(f)
run_name="ipynb"
dataset_name="MMLongBench"
dq_cfg["run-name"] = run_name
dataset = BaseDataset(dq_cfg, dataset_name)
prompt="""
You are tasked with summarizing a document page by page. You will be given:
- The current summary that has been built so far,
- The previous page (for context),
- And a new page as the current page to process.

For each new page:
1. Provide a summary of the page's main content. If the page is blank or has no meaningful content, summarize it as "no content".
2. Identify any figures or tables on the page. For each:
   - If a name or caption is provided (e.g., "Figure 1: Overview of System Architecture"), include the name.If no name is provided, leave it blank,like this: ```Figure : Overview of System Architecture```.Don't name the pictures yourself.
   - Provide a one-sentence description of the figure or table.
   - If this page does not contain any figure and table, there is no need to summarize them — only a summary of the page is required.
3. Maintain a structured output format as described below.

The final output should follow this format:
```
- Page {number}: [One-sentence summary of the page content.]
  - Figure {name}: [One-sentence summary of the figure.]
  - Table: [One-sentence summary of the table.]
```
or
```
- Page {number}: no content
  - no figure and table
```
Just provide the summary for the current new page.
"""
ds=DocumentSummarizer(dataset=dataset,prompt=prompt)
ds.summarize_document("/home/forlorin/project/DocQuest/data/MMLongBench/documents/2303.05039v2.pdf")

In [1]:
from mydatasets.BaseDataset import BaseDataset
from agents.doc_quest import DocQuestAgents
import toml  # type: ignore[import-untyped]

toml_cfg_path="config/doc_quest_config.toml"
with open(toml_cfg_path, "r") as f:
        dq_cfg = toml.load(f)
run_name="ipynb"
dataset_name="MMLongBench"
dq_cfg["run-name"] = run_name
dataset = BaseDataset(dq_cfg, dataset_name)
docQuestAgents = DocQuestAgents(dq_cfg)
sample={
        "doc_id": "0e94b4197b10096b1f4c699701570fbf.pdf",
        "doc_type": "Tutorial/Workshop",
        "question": "According to the chart on page 14 how much time was spent with family and friends in 2010?",
        "answer": "21%",
        "evidence_pages": "[14]",
        "evidence_sources": "['Chart']",
        "answer_format": "Float",
        "retrieval-query": "According to the chart on page 14, some amount of time was spent with family and friends in 2010 for unspecified reasons.",
        "retrieval-key": "chart, page 14, time spent, family and friends, 2010",
        "qwen_retrieval-key": "2010, time spent, family and friends",
        "qwen_retrieval-query": "According to the chart on page 14, some amount of time was spent with family and friends in 2010 for some reason.",
        "text-index-path-question": ".ragatouille/colbert/indexes/fsl-question-0e94b4197b10096b1f4c699701570fbf.pdf",
        "text-top-10-question": [
            9,
            3,
            8,
            2,
            10,
            11,
            0,
            7,
            6,
            13
        ],
        "text-top-10-question_score": [
            7.75390625,
            6.7890625,
            6.6875,
            6.4765625,
            6.37890625,
            6.35546875,
            6.21484375,
            6.01953125,
            5.89453125,
            5.8515625
        ],
        "image-top-10-question": [
            3,
            12,
            5,
            14,
            13,
            0,
            10,
            11,
            8,
            6
        ],
        "image-top-10-question_score": [
            7.047145843505859,
            5.965827465057373,
            5.930147171020508,
            5.903757572174072,
            5.773287773132324,
            5.701374053955078,
            5.699911594390869,
            5.638657569885254,
            5.435965538024902,
            5.39284086227417
        ]
    }

docQuestAgents.predict(sample,dataset)

### location:
{'modal': 'Image', 'location': ['Page 14', 'Figure']}
### General Agent: To determine how much time was spent with family and friends in 2010 according to the chart on page 14, let's analyze the visual information provided:

1. **Page 14 Chart**: The chart is a pie graph comparing the time spent on different activities in 2005 and 2010.
2. **Activity Breakdown**:
   - In 2010, the activity "With family and friends" is represented by a slice of the pie chart.
3. **Percentage for 2010**:
   - The percentage for "With family and friends" in 2010 is clearly marked as 21%.

Therefore, based on the pie chart from page 14, the amount of time spent with family and friends in 2010 was **21%**. 

No conflicting information or additional details are present in the other pages that would alter this conclusion. Thus, the answer is:

**Time spent with family and friends in 2010 was 21%.**
### General Critical Agent: {"text": "21%", "image": "With family and friends 21%"}
### Text Agent

'```json\n{"Answer": "21%"}\n```'